# World model for brain MRI on OASIS-2 (Google Colab)

Runs the whole project on OASIS-2: download, preprocessing, world model, classifiers, attacks and defences.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. You need a Google account with about **3 GB free on Google Drive** (only preprocessed data, checkpoints and results go there; the 18 GB raw download stays on Colab's temporary disk).
3. You must have accepted the OASIS Data Use Terms. Do not share the data or put it in a public place.

**If Colab disconnects:** run cells 1, 2 and 6 again. Finished steps are skipped, preprocessing skips finished scans and the world model continues from its last epoch.

## 1. Connect Google Drive and get the code

In [ ]:
import glob
from IPython.display import Image, display
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/wm'
if not os.path.exists(REPO):
    !git clone -q https://github.com/SUDHAVALLI-17/World-model-brain-mri-.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
!pip install -q hd-bet   # deep-learning skull stripping (runs on the GPU); used automatically

## 2. Settings (edit the Drive folder name if you like)

In [ ]:
DRIVE = '/content/drive/MyDrive/world_model_oasis2'   # results live here
RAW   = '/content/oas2'                               # temporary raw data (lost on disconnect - that's fine)
os.makedirs(DRIVE, exist_ok=True)
os.environ.update({
    'DATA':   f'{DRIVE}/data',      # preprocessed volumes (~1.3 GB, encrypted)
    'RUNS':   f'{DRIVE}/runs',      # checkpoints and results
    'MODELS': f'{DRIVE}/models',    # packaged model for the web app
    'KEY':    f'{DRIVE}/data.key',  # encryption key - keep it private
    'OASIS2_DIR': RAW,
})
print('Drive folder:', DRIVE)

## 3. Download OASIS-2 (only needed until preprocessing has finished)

On the OASIS-2 page, right-click **OAS2_RAW_PART1.tar.gz** → *Copy link address*, paste it below, and do the same for PART2.
Only the first MPRAGE of each session (`mpr-1`) is extracted, and each archive is deleted afterwards, so the disk never fills up.

In [ ]:
PART1_URL = 'PASTE LINK TO OAS2_RAW_PART1.tar.gz'
PART2_URL = 'PASTE LINK TO OAS2_RAW_PART2.tar.gz'

already = os.path.exists(f"{os.environ['DATA']}/.done/preprocess")
if already:
    print('Preprocessing already finished in an earlier session - no download needed.')
else:
    os.makedirs(RAW, exist_ok=True)
    for i, url in enumerate([PART1_URL, PART2_URL], 1):
        tgz = f'/content/part{i}.tar.gz'
        !wget -q --show-progress -c "{url}" -O {tgz}
        !tar -xzf {tgz} -C {RAW} --wildcards '*mpr-1[._]*' && rm -f {tgz}
    !find {RAW} -name 'mpr-1*.hdr' | wc -l

## 4. Demographics file

Download **Demographic Data** from the OASIS-2 page (an Excel file), then upload it with the cell below (or drag it into the Drive folder once).

In [ ]:
import glob, shutil
found = glob.glob(f'{DRIVE}/*demographics*.xls*') + glob.glob(f'{DRIVE}/oasis_longitudinal*.csv')
if not found:
    from google.colab import files
    up = files.upload()
    for name in up:
        shutil.move(name, f'{DRIVE}/{name}')
    found = [f'{DRIVE}/{n}' for n in up]
os.environ['OASIS2_CSV'] = found[0]
print('demographics:', found[0])

## 4b. Check the brain mask before anything else

Builds the scan list, then draws (with HD-BET, the deep-learning skull stripper) the skull-strip mask (red outline) on a few scans **before** cropping and resizing,
with and without the neck crop. The red line should follow the outer edge of the brain all round, include the
cerebellum and the bottom of the temporal lobes, and exclude skull, scalp, eyes and neck. Row 1 shows the raw file;
row 2 shows the oriented image (top of head up, nose to the right in the sagittal view) with the neck-crop window
in yellow. Send these images if anything looks wrong.

In [ ]:
!python src/prepare_oasis.py --oasis2_dir {RAW} --oasis2_csv "{os.environ['OASIS2_CSV']}" --out /content/index_check.csv
!cd src && python qc_mask.py --index /content/index_check.csv --n 3 --compare_neck --out {DRIVE}/qc_mask
# for comparison, the classical BET fallback:
!cd src && python qc_mask.py --index /content/index_check.csv --n 3 --skullstrip bet --out {DRIVE}/qc_mask_bet
for p in sorted(glob.glob(f'{DRIVE}/qc_mask/*.png')) + sorted(glob.glob(f'{DRIVE}/qc_mask_bet/*.png')):
    print(os.path.basename(p)); display(Image(p))

If the version **without** neck crop is the correct one, run every later cell with it:

In [ ]:
os.environ['PREP_FLAGS'] = '--no_neck_crop'   # set to '' to keep the neck crop

## 5. Quick check (about 20-30 minutes) - every stage for 1-3 epochs. Its numbers mean nothing.

In [ ]:
os.environ['SMOKE'] = '1'
smoke = dict(DATA=f'{DRIVE}/smoke/data', RUNS=f'{DRIVE}/smoke/runs', MODELS=f'{DRIVE}/smoke/models')
env = ' '.join(f'{k}={v}' for k, v in smoke.items())
!{env} TRAIN_FOLDS=1 MODELSET=core bash run_all.sh 2>&1 | grep -E "^==|INFO.*(done|ep |saved|best)|ERROR|Error" | tail -60
os.environ['SMOKE'] = '0'

Look at a few preprocessing check images (brain should be upright, skull removed). If they look wrong, stop and report it.

In [ ]:
from IPython.display import Image, display
for p in sorted(glob.glob(f'{DRIVE}/smoke/data/proc/qc/*.png'))[:4]:
    print(os.path.basename(p)); display(Image(p))

## 6. Full training

Set the size to fit your GPU time. `TRAIN_FOLDS=1 MODELSET=core` is the quickest real run (a few hours on a T4);
`TRAIN_FOLDS=5 MODELSET=all` is the full paper setup (1-2 days, over several sessions - just re-run this cell after each disconnect).

In [ ]:
TRAIN_FOLDS = 1        # 1 = fold 0 only; 5 = all folds
MODELSET    = 'core'   # 'core' or 'all'
EP_PRE, EP_CLS = 100, 40

!TRAIN_FOLDS={TRAIN_FOLDS} MODELSET={MODELSET} EP_PRE={EP_PRE} EP_CLS={EP_CLS} bash run_all.sh 2>&1 | grep -E "^==|INFO.*(done|saved|best epoch|resumed)|ERROR|Error" | tail -80

## 7. Results

In [ ]:
import json, pandas as pd
R = os.environ['RUNS']
for f in ['test/test_report.json', 'robustness/robustness_report.json', 'paper1/paper1_report.json']:
    p = f'{R}/{f}'
    if os.path.exists(p):
        print('==', f); print(json.dumps(json.load(open(p)), indent=1)[:3000])
for img in ['robustness/robustness_curve.png', 'paper1/paper1_figures.png']:
    if os.path.exists(f'{R}/{img}'):
        display(Image(f'{R}/{img}'))
print('Send these folders (numbers only, no scans) for the paper:', f'{R}/test', f'{R}/robustness', f'{R}/paper1')

## 8. Fixing weak classification: diagnose, then a controlled comparison

Run these in order **after** preprocessing and world-model pretraining (section 6). They reuse the existing
`splits.json` and checkpoints; nothing here repeats HD-BET except the optional fixed-crop step 8c.

**Keep the current `splits.json`.** Re-splitting would change the held-out test subjects, and the world model
was pretrained on everything except them.

On OASIS the labels come from CDR: CN = CDR 0, MCI = CDR 0.5 ("very mild dementia" in OASIS, not clinical
MCI), AD = CDR >= 1. Report them under the CDR names.

### 8a. Label audit + non-imaging baselines (a few minutes, CPU)

Shows how many AD **subjects** (not just scans) each validation fold has, and how well OASIS's own brain volume
(nWBV), our mask volume / eTIV, and age + sex alone separate the classes on the same folds.

In [ ]:
!python src/diagnose_labels.py --index {os.environ['DATA']}/proc/index_proc.csv --splits {os.environ['DATA']}/splits.json --out {os.environ['RUNS']}/diagnose
from IPython.display import Markdown
for f in ['label_audit.md', 'baselines.md']:
    display(Markdown(open(f"{os.environ['RUNS']}/diagnose/{f}").read()))

### 8a2. Feature-collapse check (about 10 minutes on a GPU)

Do different scans get different features? Compares a random encoder with your trained checkpoints on the
same 96 training scans, under each pooling of the 216 latent tokens. `spread/RMS` near 0.01 and `cosine` near
1 = every scan gets almost the same vector. If `tokens` differ but `attn` does not, the pooling throws the
differences away: then use the `scratch_mmbn` / `wm_ft_mmbn` experiments in 8b (`--pool meanmax --head_norm batch`).
Point the paths at your own checkpoints.

In [ ]:
R, D = os.environ['RUNS'], os.environ['DATA']
CKPTS = {
    'world_model': f'{R}/pretrain/last.pt',
    # 'scratch':  f'{R}/<your scratch classifier run>/best.pt',
    # 'no_aug':   f'{R}/<your no-augmentation classifier run>/best.pt',
}
args = ' '.join(f'--ckpt {k}={v}' for k, v in CKPTS.items() if os.path.exists(v))
!python src/feature_spread.py --index {D}/proc/index_proc.csv --splits {D}/splits.json --fold 0 --key_file {os.environ['KEY']} --n 96 {args} --out {R}/feature_spread
display(Markdown(open(f'{R}/feature_spread/feature_spread.md').read()))

### 8b. Controlled cross-validation

Same folds, same settings, several seeds, one change at a time: `scratch` (no pretraining), `wm_ft` (world model,
fine-tuned), `wm_probe` (world model frozen, only the head trains) and `wm_ft_vol` (+ brain volume / eTIV as an
input). Image-only, no label smoothing, no flip averaging, fixed epochs, checkpoint = lowest class-balanced
validation loss.

It starts with two **sanity checks** on fold 0 (`sanity/sanity.md`):
- **memorise 64 scans** and score on the same scans: must reach about 1.0. If not, the model cannot fit
  even a small set (collapse, learning rate, input data) and more data or tuning will not help;
- **shuffled training labels**, real validation labels: must be at or below chance. Clearly above chance
  means the label reaches the model another way (leakage or a join bug).

Cost: experiments x seeds x 5 folds runs. Start small (one seed, two experiments), then add the rest; finished
runs are skipped when you re-run the cell.

In [ ]:
TASK  = 'cdr_binary'            # 'cdr_binary' (CDR 0 vs >=0.5), 'cn_vs_ad' (CDR 0 vs >=1) or '3class'
SEEDS = '42'                    # then '42 43 44'
EXPS  = 'scratch_noimag scratch_mmbn_noimag wm_ft wm_ft_mmbn'   # then add 'wm_probe wm_ft_vol'
EP_CLS = 40
!TASK={TASK} SEEDS="{SEEDS}" EXPS="{EXPS}" EP_CLS={EP_CLS} bash run_cv.sh 2>&1 | grep -E "^==|WARNING|best epoch|Error|experiment|verdict|scratch|wm_" | tail -60
display(Markdown(open(f"{os.environ['RUNS']}/cv/{TASK}/sanity/sanity.md").read()))
display(Markdown(open(f"{os.environ['RUNS']}/cv/{TASK}/report/cv_report.md").read()))

### 8c. Optional: fixed-size crop (keeps whole-brain atrophy in the image)

The default crop scales every brain to fill the cube, which removes how much the brain has shrunk. This
re-crops every brain in the same 192 mm box. It runs HD-BET once more, but caches the skull-stripped images
in `stage1/`, so later crop or size changes take minutes. The world model was pretrained on the old crop, so
`scratch_fixed` vs `scratch` is the clean comparison.

In [ ]:
D = os.environ['DATA']
!python src/preprocess.py --index {D}/index.csv --out_dir {D}/proc_fixed --target 96 --spacing 1.5 --skullstrip auto --encrypt --key_file {os.environ['KEY']} --resume --qc --crop fixed --crop_mm 192 --stage1_dir {D}/stage1 {os.environ.get('PREP_FLAGS', '')} 2>&1 | grep -E "INFO.*(done|saved)|SKIPPED|WARNING|ERROR" | tail -20
!TASK={TASK} SEEDS="{SEEDS}" EXPS="scratch" FIXED_EXPS=scratch_fixed INDEX_FIXED={D}/proc_fixed/index_proc.csv EP_CLS={EP_CLS} bash run_cv.sh 2>&1 | grep -E "^==|WARNING|Error|experiment|scratch|wm_" | tail -40
display(Markdown(open(f"{os.environ['RUNS']}/cv/{TASK}/report/cv_report.md").read()))

**How to read the report:** a change helps only if its `AUC - scratch` interval excludes 0 and it holds
across seeds. The image model should beat the `age_sex` baseline from 8a and approach `nWBV (OASIS)`.
Only after a configuration wins here should it be trained with the adversarial defences and scored once on
the test set (`evaluate.py`).